# Decision Tree Classifier

7-class classifier. A single shallow tree — smallest possible tree-based model (a plain if/else chain once exported).

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
%pip install pandas numpy scikit-learn joblib openpyxl matplotlib skl2onnx onnx

In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree


In [ ]:
# ==========================================================
# CELL 2 — Decision Tree configuration
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_PATH = r"../../data/Motor_data_model"
SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s", "PWM", "Irms", "Vrms",
    "Current_Variance", "Current_Crest_Factor",
    "Vib_RMS", "Vib_Min", "Vib_Q1", "Vib_Median",
    "Vib_Q3", "Vib_Max", "Vib_Variance", "Vib_Skewness",
]

FINAL_LABELS = [
    "motor_off", "turning_off", "turning_on", "normal",
    "electrical_fault", "mechanical_fault", "thermal_fault",
]

LABEL_TO_ID = {label: index for index, label in enumerate(FINAL_LABELS)}

N_FEATURES = len(FEATURE_COLUMNS)
N_CLASSES = len(FINAL_LABELS)
WINDOW_LEN = 30
WINDOW_STEP = 10
INPUT_SIZE = WINDOW_LEN * N_FEATURES
ROW_PERIOD_SEC = 0.015
N_SPLITS = 5

# Regularization to reduce overfitting.
MAX_DEPTH = 50
MIN_SAMPLES_SPLIT = 20
MIN_SAMPLES_LEAF = 8

OUTPUT_DIR = "decision_tree_output"
CV_DIR = os.path.join(OUTPUT_DIR, "cross_validation")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_model")
MANUAL_TEST_DIR = os.path.join(OUTPUT_DIR, "manual_test_results")

for folder in [CV_DIR, FINAL_DIR, MANUAL_TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Model: Decision Tree")
print("Input:", WINDOW_LEN, "x", N_FEATURES)
print("Classes:", FINAL_LABELS)


In [ ]:
# ==========================================================
# CELL 3 — map each file path to one supervised class
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def label_for(path):
    path_text = normalized_path(path)
    stem = Path(path).stem.lower()

    if any(x in path_text for x in [
        "motor_being_off", "turning_off", "transition_off", "off_spikes"
    ]):
        return "turning_off"

    if any(x in path_text for x in [
        "motor_being_on", "turning_on", "transition_on", "on_spikes"
    ]):
        return "turning_on"

    if "/motor_off/" in path_text or "motor_off" in stem:
        return "motor_off"

    if any(x in path_text for x in [
        "electrical fault", "electrical_fault", "abnormal_current"
    ]):
        return "electrical_fault"

    if any(x in path_text for x in [
        "mechanical fault", "mechanical_fault", "abnormal_vibration"
    ]):
        return "mechanical_fault"

    if any(x in path_text for x in [
        "thermal fault", "thermal_fault", "abnormal_temp"
    ]):
        return "thermal_fault"

    if "/normal/" in path_text or stem.startswith("normal_") or stem == "normal":
        return "normal"

    return None

In [ ]:
# ==========================================================
# CELL 4 — discover and read CSV / Excel files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(os.path.join(base_path, "**", "*.xlsx"), recursive=True)
    files += glob.glob(os.path.join(base_path, "**", "*.csv"), recursive=True)
    return sorted(set(files))


def read_one(path):
    path = Path(path)
    df = pd.read_excel(path) if path.suffix.lower() == ".xlsx" else pd.read_csv(path)

    missing_columns = [c for c in FEATURE_COLUMNS if c not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in {path.name}: {missing_columns}")

    clean = df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
    clean = clean / SCALE_DIVISOR
    clean = clean.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(f"{path.name} has only {len(clean)} valid rows.")

    return clean

In [ ]:
# ==========================================================
# CELL 5 — load labeled sessions
# ==========================================================
files = discover_files(BASE_PATH)
if not files:
    raise RuntimeError(f"No CSV or Excel files found under: {BASE_PATH}")

sessions = []
for file_path in files:
    label = label_for(file_path)
    if label is None:
        print("SKIP unrecognized:", os.path.relpath(file_path, BASE_PATH))
        continue

    try:
        df = read_one(file_path)
    except Exception as error:
        print("SKIP read error:", os.path.relpath(file_path, BASE_PATH), "|", error)
        continue

    sessions.append({
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(file_path, BASE_PATH),
        "label": label,
        "class_id": LABEL_TO_ID[label],
        "rows": len(df),
    })

if not sessions:
    raise RuntimeError("No labeled sessions were loaded.")

class_counts = pd.Series([s["label"] for s in sessions]).value_counts().reindex(
    FINAL_LABELS, fill_value=0
)
print("\nFILES PER CLASS")
print(class_counts)

missing_classes = [label for label in FINAL_LABELS if class_counts[label] == 0]
if missing_classes:
    raise RuntimeError(f"No files found for classes: {missing_classes}")

In [ ]:
# ==========================================================
# CELL 6 — create flattened 30 × 14 windows
# ==========================================================
def make_windows_from_array(X_raw):
    starts = np.arange(
        0,
        len(X_raw) - WINDOW_LEN + 1,
        WINDOW_STEP,
        dtype=np.int64,
    )
    if len(starts) == 0:
        return None, None

    windows = np.stack([
        X_raw[start:start + WINDOW_LEN]
        for start in starts
    ]).astype(np.float32)

    return windows.reshape(len(windows), INPUT_SIZE), starts


def build_dataset_from_sessions(selected_sessions):
    all_windows, all_labels, all_files, all_starts = [], [], [], []

    for session in selected_sessions:
        windows, starts = make_windows_from_array(session["X"])
        if windows is None:
            continue

        all_windows.append(windows)
        all_labels.extend([session["class_id"]] * len(windows))
        all_files.extend([session["file"]] * len(windows))
        all_starts.extend(starts.tolist())

    if not all_windows:
        raise RuntimeError("No windows were created.")

    return (
        np.concatenate(all_windows, axis=0),
        np.asarray(all_labels, dtype=np.int64),
        np.asarray(all_files, dtype=object),
        np.asarray(all_starts, dtype=np.int64),
    )

In [ ]:
# ==========================================================
# CELL 7 — flattened feature names
# ==========================================================
FLAT_FEATURE_NAMES = [
    f"t{time_index:02d}_{feature_name}"
    for time_index in range(WINDOW_LEN)
    for feature_name in FEATURE_COLUMNS
]

In [ ]:
# ==========================================================
# CELL 8 — build Decision Tree classifier
# ==========================================================
def build_classifier():
    return DecisionTreeClassifier(
        criterion="gini",
        splitter="best",
        max_depth= None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features= None,
        class_weight="balanced",
        random_state=42,
    )

print(build_classifier().get_params())


In [ ]:
# ==========================================================
# CELL 9 — five-fold file-wise cross-validation
# ==========================================================
cv_table = pd.DataFrame({
    "label": [s["class_id"] for s in sessions],
    "group": [s["file"] for s in sessions],
})

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

cv_metrics = []
all_true = []
all_pred = []
all_window_predictions = []

for fold, (train_index, test_index) in enumerate(
    splitter.split(cv_table, cv_table["label"], groups=cv_table["group"]),
    start=1,
):
    print("\n" + "=" * 60)
    print("FOLD", fold)
    print("=" * 60)

    train_sessions = [sessions[index] for index in train_index]
    test_sessions = [sessions[index] for index in test_index]

    X_train, y_train, _, _ = build_dataset_from_sessions(train_sessions)
    X_test, y_test, test_files, test_starts = build_dataset_from_sessions(test_sessions)

    model = build_classifier()
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)
    confidence = np.max(probabilities, axis=1)

    all_true.extend(y_test.tolist())
    all_pred.extend(predictions.tolist())

    fold_metrics = {
        "fold": fold,
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "macro_precision": precision_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "tree_depth": model.get_depth(),
        "leaf_count": model.get_n_leaves(),
    }
    cv_metrics.append(fold_metrics)

    all_window_predictions.append(pd.DataFrame({
        "fold": fold,
        "file": test_files,
        "start_row": test_starts,
        "true_class_id": y_test,
        "true_label": [FINAL_LABELS[index] for index in y_test],
        "pred_class_id": predictions,
        "pred_label": [FINAL_LABELS[index] for index in predictions],
        "confidence": confidence,
    }))

    print(pd.Series(fold_metrics))

In [ ]:
# ==========================================================
# CELL 10 — combined cross-validation report
# ==========================================================
cv_metrics_df = pd.DataFrame(cv_metrics)
cv_predictions_df = pd.concat(all_window_predictions, ignore_index=True)

cv_metrics_df.to_csv(os.path.join(CV_DIR, "cv_metrics.csv"), index=False)
cv_predictions_df.to_csv(
    os.path.join(CV_DIR, "cv_window_predictions.csv"), index=False
)

all_true = np.asarray(all_true, dtype=np.int64)
all_pred = np.asarray(all_pred, dtype=np.int64)

overall_metrics = {
    "accuracy": accuracy_score(all_true, all_pred),
    "balanced_accuracy": balanced_accuracy_score(all_true, all_pred),
    "macro_precision": precision_score(
        all_true, all_pred, average="macro", zero_division=0
    ),
    "macro_recall": recall_score(
        all_true, all_pred, average="macro", zero_division=0
    ),
    "macro_f1": f1_score(
        all_true, all_pred, average="macro", zero_division=0
    ),
}

print("\nOVERALL CROSS-VALIDATION RESULTS")
print(pd.Series(overall_metrics))
print(classification_report(
    all_true,
    all_pred,
    labels=np.arange(N_CLASSES),
    target_names=FINAL_LABELS,
    zero_division=0,
))

pd.DataFrame([overall_metrics]).to_csv(
    os.path.join(CV_DIR, "overall_metrics.csv"), index=False
)

ConfusionMatrixDisplay(
    confusion_matrix(all_true, all_pred, labels=np.arange(N_CLASSES)),
    display_labels=FINAL_LABELS,
).plot(values_format="d", xticks_rotation=45)

plt.title("Decision Tree Cross-Validation")
plt.tight_layout()
plt.savefig(os.path.join(CV_DIR, "confusion_matrix.png"), dpi=180)
plt.show()

In [ ]:
# ==========================================================
# CELL 11 — train final Decision Tree on all sessions
# ==========================================================
X_all, y_all, _, _ = build_dataset_from_sessions(sessions)
final_model = build_classifier()
final_model.fit(X_all, y_all)

print("Final tree depth:", final_model.get_depth())
print("Final leaf count:", final_model.get_n_leaves())

In [ ]:
# ==========================================================
# CELL 12 — save model, labels, metadata, and rules
# ==========================================================
MODEL_PATH = os.path.join(FINAL_DIR, "decision_tree_classifier.pkl")
LABELS_PATH = os.path.join(FINAL_DIR, "decision_tree_labels.txt")
METADATA_PATH = os.path.join(FINAL_DIR, "decision_tree_metadata.csv")
RULES_PATH = os.path.join(FINAL_DIR, "decision_tree_rules.txt")

joblib.dump(final_model, MODEL_PATH)

with open(LABELS_PATH, "w", encoding="utf-8") as file:
    file.write("\n".join(FINAL_LABELS))

pd.DataFrame({
    "flat_feature_id": np.arange(INPUT_SIZE),
    "flat_feature_name": FLAT_FEATURE_NAMES,
}).to_csv(METADATA_PATH, index=False)

with open(RULES_PATH, "w", encoding="utf-8") as file:
    file.write(export_text(final_model, feature_names=FLAT_FEATURE_NAMES))

print("Saved:", MODEL_PATH)
print("Saved:", LABELS_PATH)
print("Saved:", METADATA_PATH)
print("Saved:", RULES_PATH)


In [ ]:
# ==========================================================
# CELL 13 — feature importance
# ==========================================================
importance_df = pd.DataFrame({
    "flat_feature_name": FLAT_FEATURE_NAMES,
    "importance": final_model.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

importance_df.to_csv(
    os.path.join(FINAL_DIR, "feature_importance.csv"), index=False
)
print("\nTOP 30 TREE FEATURES")
print(importance_df.head(30))


In [ ]:
# ==========================================================
# CELL 14 — tree visualization
# ==========================================================
plt.figure(figsize=(28, 14))
plot_tree(
    final_model,
    feature_names=FLAT_FEATURE_NAMES,
    class_names=FINAL_LABELS,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=7,
)
plt.title("Decision Tree — first 3 levels")
plt.tight_layout()
plt.savefig(
    os.path.join(FINAL_DIR, "decision_tree_top_levels.png"), dpi=180
)
plt.show()


In [ ]:
# ==========================================================
# CELL 15 — export ONNX
# ==========================================================
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

ONNX_PATH = os.path.join(FINAL_DIR, "decision_tree_classifier.onnx")
onnx_model = convert_sklearn(
    final_model,
    initial_types=[("float_input", FloatTensorType([None, INPUT_SIZE]))],
    target_opset=13,
)
with open(ONNX_PATH, "wb") as file:
    file.write(onnx_model.SerializeToString())
    
print("Saved:", ONNX_PATH)

In [ ]:
# ==========================================================
# Generate and download Decision Tree parameter header
# ==========================================================
from pathlib import Path


HEADER_PATH = Path(FINAL_DIR) / "decision_tree_params.h"

labels_c = ",\n".join(
    f'    "{label}"' for label in FINAL_LABELS
)

features_c = ",\n".join(
    f'    "{feature}"' for feature in FEATURE_COLUMNS
)

header = f"""#ifndef DECISION_TREE_PARAMS_H
#define DECISION_TREE_PARAMS_H

#define DT_WINDOW_LEN       {WINDOW_LEN}U
#define DT_WINDOW_STEP      {WINDOW_STEP}U
#define DT_N_FEATURES       {N_FEATURES}U
#define DT_INPUT_SIZE       {INPUT_SIZE}U
#define DT_N_CLASSES        {N_CLASSES}U
#define DT_ROW_PERIOD_SEC   {ROW_PERIOD_SEC:.10f}f

/*
 * The model expects physical FLOAT32 feature values.
 * Do not divide live STM32 values by 1,000,000.
 */
#define DT_EXTERNAL_SCALING  0U
#define DT_LIVE_DIVISION     0U

static const char *const DT_LABELS[DT_N_CLASSES] = {{
{labels_c}
}};

static const char *const DT_FEATURE_NAMES[DT_N_FEATURES] = {{
{features_c}
}};

#endif /* DECISION_TREE_PARAMS_H */
"""

HEADER_PATH.write_text(header, encoding="utf-8")

print("Created:", HEADER_PATH)


In [ ]:
# ==========================================================
# CELL 16 — export complete direct C header
# ==========================================================
HEADER_PATH = os.path.join(FINAL_DIR, "decision_tree_classifier_model.h")
tree = final_model.tree_
feature = tree.feature
threshold = tree.threshold
children_left = tree.children_left
children_right = tree.children_right
predicted_class = np.argmax(tree.value[:, 0, :], axis=1).astype(np.int32)


def write_c_array(file, c_type, name, values, values_per_line=12):
    values = np.asarray(values).reshape(-1)
    file.write(f"static const {c_type} {name}[{len(values)}] = {{\n")

    for start in range(0, len(values), values_per_line):
        chunk = values[start:start + values_per_line]
        if "float" in c_type:
            line = ", ".join(f"{float(value):.9g}f" for value in chunk)
        else:
            line = ", ".join(str(int(value)) for value in chunk)
        file.write(f"    {line},\n")

    file.write("};\n\n")


with open(HEADER_PATH, "w", encoding="utf-8") as file:
    file.write("#ifndef DECISION_TREE_CLASSIFIER_MODEL_H\n")
    file.write("#define DECISION_TREE_CLASSIFIER_MODEL_H\n\n")
    file.write("#include <stdint.h>\n\n")

    file.write(f"#define DT_WL {WINDOW_LEN}\n")
    file.write(f"#define DT_NF {N_FEATURES}\n")
    file.write(f"#define DT_INPUT {INPUT_SIZE}\n")
    file.write(f"#define DT_N_CLASSES {N_CLASSES}\n")
    file.write(f"#define DT_NODE_COUNT {tree.node_count}\n")
    file.write(f"#define DT_DIV {SCALE_DIVISOR:.9g}f\n\n")

    write_c_array(file, "int16_t", "DT_FEATURE", feature)
    write_c_array(file, "float", "DT_THRESHOLD", threshold)
    write_c_array(file, "int16_t", "DT_LEFT", children_left)
    write_c_array(file, "int16_t", "DT_RIGHT", children_right)
    write_c_array(file, "uint8_t", "DT_CLASS", predicted_class)

    file.write(r'''
static inline int decision_tree_predict(
    const float raw[DT_WL][DT_NF])
{
    int node = 0;

    while (DT_LEFT[node] != -1)
    {
        int feature_id = DT_FEATURE[node];
        int time_index = feature_id / DT_NF;
        int feature_index = feature_id % DT_NF;

        float physical =
            raw[time_index][feature_index] / DT_DIV;

        if (physical <= DT_THRESHOLD[node])
        {
            node = DT_LEFT[node];
        }
        else
        {
            node = DT_RIGHT[node];
        }
    }

    return (int)DT_CLASS[node];
}

#endif
''')

print("Saved:", HEADER_PATH)
print("Header size KB:", round(os.path.getsize(HEADER_PATH) / 1024, 2))

In [ ]:
# ==========================================================
# CELL 17 — verify direct tree traversal
# ==========================================================
def direct_python_tree_predict(flattened_window):
    node = 0
    while children_left[node] != -1:
        feature_id = feature[node]
        if flattened_window[feature_id] <= threshold[node]:
            node = children_left[node]
        else:
            node = children_right[node]
    return int(predicted_class[node])

number_check = min(100, len(X_all))
sklearn_predictions = final_model.predict(X_all[:number_check])
direct_predictions = np.asarray([
    direct_python_tree_predict(window)
    for window in X_all[:number_check]
])

if not np.array_equal(sklearn_predictions, direct_predictions):
    raise RuntimeError("Direct traversal does not match sklearn.")

print("Direct C-style traversal verified on", number_check, "windows.")

In [ ]:
# ==========================================================
# CELL 18 — manual classification test
# ==========================================================
def manual_classification_prediction(test_file):
    model = joblib.load(MODEL_PATH)
    test_df = read_one(test_file)
    X_test, starts = make_windows_from_array(
        test_df.to_numpy(dtype=np.float32)
    )

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)
    confidence = np.max(probabilities, axis=1)

    result = pd.DataFrame({
        "window": np.arange(len(X_test)),
        "start_row": starts,
        "time_sec": (starts + WINDOW_LEN / 2.0) * ROW_PERIOD_SEC,
        "pred_class_id": predictions,
        "pred_label": [FINAL_LABELS[index] for index in predictions],
        "confidence": confidence,
    })

    for class_id, label in enumerate(FINAL_LABELS):
        result[f"probability_{label}"] = probabilities[:, class_id]

    majority_class_id = int(result["pred_class_id"].value_counts().idxmax())
    majority_label = FINAL_LABELS[majority_class_id]
    stem = Path(test_file).stem

    output_csv = os.path.join(
        MANUAL_TEST_DIR,
        f"manual_test_{stem}.csv",
    )
    result.to_csv(output_csv, index=False)

    print("\n" + "=" * 60)
    print("MANUAL DECISION TREE TEST")
    print("=" * 60)
    print("File:", test_file)
    print("Total windows:", len(result))
    print("Majority prediction:", majority_label)
    print("Mean confidence:", float(np.mean(confidence)))
    print("Saved CSV:", output_csv)
    print("\nWindow predictions:")
    print(result["pred_label"].value_counts())

    plt.figure(figsize=(12, 4))
    plt.step(
        result["time_sec"],
        result["pred_class_id"],
        where="mid",
    )
    plt.yticks(np.arange(N_CLASSES), FINAL_LABELS)
    plt.xlabel("Time [s]")
    plt.ylabel("Predicted class")
    plt.title(f"Decision Tree — {stem}")
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            MANUAL_TEST_DIR,
            f"manual_test_{stem}_timeline.png",
        ),
        dpi=150,
    )
    plt.show()

    return result

In [ ]:
# ==========================================================
# CELL 19 — select manual test file
# ==========================================================
if __name__ == "__main__":
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    TEST_FILE = filedialog.askopenfilename(
        title="Select motor CSV or Excel file",
        filetypes=[
            ("Excel files", "*.xlsx"),
            ("CSV files", "*.csv"),
            ("All files", "*.*"),
        ],
    )

    root.destroy()

    if TEST_FILE:
        manual_result = manual_classification_prediction(TEST_FILE)
    else:
        print("No file selected.")